# Task 2.2: End-to-End Model Pipeline

This notebook demonstrates baseline creation, EDA, feature engineering, modeling, evaluation, and hyperparameter tuning for the California Housing dataset.


In [94]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid")


## Step 2: Prep & Split
Load data, check missing values, handle outliers, split, and scale.


In [95]:
housing_bundle = fetch_california_housing()
X, y = housing_bundle.data, housing_bundle.target
feature_names = housing_bundle.feature_names

dfCalifornia = pd.DataFrame(X, columns=feature_names)
dfCalifornia['MedHouseVal'] = y

print("Missing values:")
print(dfCalifornia.isna().sum())

print("\nDataset Info:")
print(dfCalifornia.info())


Missing values:
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None


In [96]:
# Outlier Removal using IQR method
Q1 = dfCalifornia.quantile(0.25)
Q3 = dfCalifornia.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter rows where all values are within bounds
mask = ~((dfCalifornia < lower_bound) | (dfCalifornia > upper_bound)).any(axis=1)
df_clean = dfCalifornia[mask].copy()

# Drop duplicates just in case
df_clean = df_clean.drop_duplicates()

print(f"Original shape: {dfCalifornia.shape}")
print(f"Cleaned shape: {df_clean.shape}")

# Separate features and target
X_clean = df_clean.drop('MedHouseVal', axis=1).values
y_clean = df_clean['MedHouseVal'].values


Original shape: (20640, 9)
Cleaned shape: (16312, 9)


In [97]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Step 1: Baseline First
Compute a dumb baseline (always predict the mean) to prove our real models are learning.


In [98]:
dummy_regr = DummyRegressor(strategy="mean")
dummy_regr.fit(X_train_scaled, y_train)

y_pred_dummy = dummy_regr.predict(X_test_scaled)

rmse_dummy = math.sqrt(mean_squared_error(y_test, y_pred_dummy))
r2_dummy = r2_score(y_test, y_pred_dummy)
mae_dummy = mean_absolute_error(y_test, y_pred_dummy)

print("Dummy Baseline Metrics:")
print(f"RMSE: {rmse_dummy:.4f}")
print(f"R2 Score: {r2_dummy:.4f}")
print(f"MAE Score: {mae_dummy:.4f}")


Dummy Baseline Metrics:
RMSE: 0.9355
R2 Score: -0.0012
MAE Score: 0.7577


## Step 3: Train and Compare Algorithms
Train Linear Regression and a stronger tree-based model (Random Forest).


In [99]:
# Linear Regression
lrModel = LinearRegression()
lrModel.fit(X_train_scaled, y_train)

y_pred_lr = lrModel.predict(X_test_scaled)
rmse_lr = math.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print("Linear Regression Metrics:")
print(f"RMSE: {rmse_lr:.4f}")
print(f"R2 Score: {r2_lr:.4f}")
print(f"MAE Score: {mae_lr:.4f}")


Linear Regression Metrics:
RMSE: 0.5626
R2 Score: 0.6379
MAE Score: 0.4236


In [100]:
# Random Forest
rfBaseModel = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rfBaseModel.fit(X_train, y_train)

y_pred_rf = rfBaseModel.predict(X_test)
rmse_rf = math.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print("Random Forest Metrics:")
print(f"RMSE: {rmse_rf:.4f}")
print(f"R2 Score: {r2_rf:.4f}")
print(f"MAE Score: {mae_rf:.4f}")


Random Forest Metrics:
RMSE: 0.4224
R2 Score: 0.7959
MAE Score: 0.2908


## Step 4: Evaluate Honestly
Comparison of Single-Split Test Scores


In [101]:
metrics_df = pd.DataFrame({
    'Model': ['Mean Baseline', 'Linear Regression', 'Random Forest'],
    'MAE': [mae_dummy, mae_lr, mae_rf],
    'RMSE': [rmse_dummy, rmse_lr, rmse_rf],
    'R2': [r2_dummy, r2_lr, r2_rf]
})
print(metrics_df.to_markdown(index=False))


| Model             |      MAE |     RMSE |          R2 |
|:------------------|---------:|---------:|------------:|
| Mean Baseline     | 0.757672 | 0.935487 | -0.00116228 |
| Linear Regression | 0.423553 | 0.562603 |  0.637896   |
| Random Forest     | 0.290836 | 0.422435 |  0.79585    |


## Step 5: Improve and Validate
Perform feature engineering, cross validation, and hyperparameter tuning.


In [102]:
# 1. Feature Engineering
df_clean["BedroomsPerRoom"] = df_clean["AveBedrms"] / df_clean["AveRooms"]
df_clean["RoomsPerOccupant"] = df_clean["AveRooms"] / df_clean["AveOccup"]

X_new = df_clean.drop('MedHouseVal', axis=1).values
y_new = df_clean['MedHouseVal'].values

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_new, y_new, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X_train_scaled2 = scaler2.fit_transform(X_train2)
X_test_scaled2 = scaler2.transform(X_test2)


In [103]:
# Function to compute 5-Fold CV Average RMSE
def get_cv_rmse(model, X_cv, y_cv):
    scores = cross_val_score(model, X_cv, y_cv, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    rmse_scores = np.sqrt(-scores)
    return rmse_scores.mean()

print("Calculating 5-Fold CV Scores...")
cv_dummy = get_cv_rmse(dummy_regr, X_train_scaled2, y_train2)
cv_lr = get_cv_rmse(LinearRegression(), X_train_scaled2, y_train2)
cv_rf_default = get_cv_rmse(RandomForestRegressor(n_estimators=100, random_state=42), X_train, y_train) # Without Feature Engineering
cv_rf_feat = get_cv_rmse(RandomForestRegressor(n_estimators=100, random_state=42), X_train2, y_train2) # With Feature Engineering

print(f"Dummy CV RMSE: {cv_dummy:.4f}")
print(f"LR CV RMSE: {cv_lr:.4f}")
print(f"RF (No Feat Eng) CV RMSE: {cv_rf_default:.4f}")
print(f"RF (With Feat Eng) CV RMSE: {cv_rf_feat:.4f}")

Calculating 5-Fold CV Scores...
Dummy CV RMSE: 0.9460
LR CV RMSE: 0.5619
RF (No Feat Eng) CV RMSE: 0.4502
RF (With Feat Eng) CV RMSE: 0.4508


In [104]:
# Function to compute 5-Fold CV Average RMSE
def get_cv_rmse(model, X_cv, y_cv):
    scores = cross_val_score(model, X_cv, y_cv, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    rmse_scores = np.sqrt(-scores)
    return rmse_scores.mean()

print("Calculating 5-Fold CV Scores:")
cv_dummy1 = get_cv_rmse(dummy_regr, X_train_scaled, y_train)
cv_lr1 = get_cv_rmse(LinearRegression(), X_train_scaled, y_train)

print(f"Dummy CV RMSE: {cv_dummy:.4f}")
print(f"LR CV RMSE: {cv_lr:.4f}")

Calculating 5-Fold CV Scores:
Dummy CV RMSE: 0.9460
LR CV RMSE: 0.5619


In [105]:
# 2. Hyperparameter Tuning using RandomizedSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rfBaseRSCV = RandomForestRegressor(random_state=42)
rfRSCV_Model = RandomizedSearchCV(
    estimator=rfBaseRSCV, 
    param_distributions=param_grid, 
    n_iter=10,
    cv=3,
    scoring='neg_mean_squared_error', 
    random_state=42,
    n_jobs=-1
)

print("Running RandomizedSearchCV (this might take a moment)...")
rfRSCV_Model.fit(X_train, y_train)
print(f"Best Parameters Found: {rfRSCV_Model.best_params_}")

# CV score of best model
best_rf_model = rfRSCV_Model.best_estimator_
cv_rf_tuned = get_cv_rmse(best_rf_model, X_train2, y_train2)
print(f"Tuned RF CV RMSE: {cv_rf_tuned:.4f}")


Running RandomizedSearchCV (this might take a moment)...
Best Parameters Found: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': None}
Tuned RF CV RMSE: 0.4500


## Final Evaluation
Now that our model is fully tuned, we evaluate it one last time on the `X_test` data that it has never seen during training or tuning. This is our brutally honest real-world metric.


In [106]:
final_predictions = best_rf_model.predict(X_test)
final_test_rmse = math.sqrt(mean_squared_error(y_test, final_predictions))
final_test_r2 = r2_score(y_test2, final_predictions)

print("FINAL TEST SET METRICS:")
print(f"Final Test RMSE: {final_test_rmse:.4f}")
print(f"Final Test R2: {final_test_r2:.4f}")

FINAL TEST SET METRICS:
Final Test RMSE: 0.4211
Final Test R2: 0.7971


In [107]:
# Save the final model
joblib.dump(best_rf_model, 'final_model.pkl')
print("Model successfully saved to 'final_model.pkl'")


Model successfully saved to 'final_model.pkl'


### Experiment Log

| Experiment | Description | 5-Fold CV Average RMSE |
| :--- | :--- | :--- |
| **Baseline LR** | Basic Linear Regression | 0.5619 |
| **Baseline RF** | Default Random Forest | 0.4502 |
| **RF + Feature Eng** | Added `BedroomsPerRoom` and `RoomsPerOccupant` | 0.4508 |
| **RF + Tuning** | Tuned using RandomizedSearchCV | 0.4500 |